In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

DATA_DIR = Path("четкие датасеты")
OUTPUT_FILE = DATA_DIR / "merged_culture_dataset.jsonl"

#берем все файлы из папки
input_files = list(DATA_DIR.glob("*.jsonl")) + list(DATA_DIR.glob("*.json"))

print("Найдено файлов:", len(input_files))
for f in input_files:
    print("-", f.name)

def norm_text(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def normalize_answer(ans):
    if ans is None:
        return None
    s = str(ans).strip().lower()
    if s in {"yes", "true", "1"}:
        return "YES"
    if s in {"no", "false", "0"}:
        return "NO"
    return ans

# очистка повторов
def dedupe_key(row):
    q = norm_text(row.get("question"))
    pd = str(row.get("prediction_date") or "").strip()
    return (q, pd)


def row_score(row):
    score = 0
    if row.get("answer") in {"YES", "NO"}:
        score += 10
    if row.get("resolution_reasoning"):
        score += 3
    if row.get("resolution_date"):
        score += 2
    if row.get("prompt"):
        score += 1
    return score


def load_any_json_file(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".jsonl":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            return [data]

        raise ValueError(f"Неожиданный формат JSON в файле {path.name}: {type(data)}")

    raise ValueError(f"Неподдерживаемый формат файла: {path.name}")


merged = {}
bad_files = []
total_rows = 0

for file_path in input_files:
    try:
        rows = load_any_json_file(file_path)
    except Exception as e:
        print(f"Ошибка чтения {file_path.name}: {e}")
        bad_files.append(file_path.name)
        continue

    print(f"\n{file_path.name}: {len(rows)} строк")

    for row in rows:
        if not isinstance(row, dict):
            continue

        total_rows += 1
        row["answer"] = normalize_answer(row.get("answer"))

        key = dedupe_key(row)

        if key not in merged:
            merged[key] = row
        else:
            if row_score(row) > row_score(merged[key]):
                merged[key] = row

final_rows = list(merged.values())

final_rows.sort(key=lambda x: (
    str(x.get("subcategory") or ""),
    str(x.get("prediction_date") or ""),
    str(x.get("question") or "")
))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for row in final_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Всего строк прочитано:", total_rows)
print("После объединения и dedupe:", len(final_rows))
print("Плохие файлы:", bad_files)
print("Файл сохранён:", OUTPUT_FILE)

cnt = Counter(r.get("answer") for r in final_rows)
print("Статистика answer:", cnt)

Найдено файлов: 15
- 1.jsonl
- 2.jsonl
- 3.jsonl
- 5.jsonl
- датасет 1 911.jsonl
- датасет 2.jsonl
- датасет 3.jsonl
- датасет 4 144.jsonl
- culture_dataset_final_1.json
- culture_dataset_final_2.json
- culture_dataset_final_3.json
- culture_dataset_final_4.json
- culture_dataset_final_5.json
- culture_dataset_final_6.json
- culture_dataset_final_7.json

1.jsonl: 139 строк

2.jsonl: 218 строк

3.jsonl: 295 строк

5.jsonl: 307 строк

датасет 1 911.jsonl: 911 строк

датасет 2.jsonl: 38 строк

датасет 3.jsonl: 79 строк

датасет 4 144.jsonl: 144 строк

culture_dataset_final_1.json: 37 строк

culture_dataset_final_2.json: 287 строк

culture_dataset_final_3.json: 363 строк

culture_dataset_final_4.json: 308 строк

culture_dataset_final_5.json: 405 строк

culture_dataset_final_6.json: 555 строк

culture_dataset_final_7.json: 633 строк

Готово.
Всего строк прочитано: 4719
После объединения и dedupe: 1996
Плохие файлы: []
Файл сохранён: четкие датасеты\merged_culture_dataset.jsonl
Статистика an